# Imports

In [1]:
import dataclasses
from dataclasses import dataclass
import json
import time
from typing import Callable, Self

from kafka import KafkaProducer
import pandas as pd

# Models

In [2]:
# import dataclasses
# from dataclasses import dataclass
# import json
# from typing import Callable, Self

# import pandas as pd

In [3]:
@dataclass
class Ride:
    lpep_pickup_datetime: str
    lpep_dropoff_datetime: str
    PU_locationID: int
    DO_locationID: int
    passenger_count: int | None  # this feature has some NaN values
    trip_distance: float
    tip_amount: float
    total_amount: float

    @classmethod
    def from_row(cls, row: pd.Series) -> Self:
        passenger_count_value = row['passenger_count']
        return cls(
            # lpep_pickup_datetime = int(row['lpep_pickup_datetime'].timestamp() * 1_000),
            # lpep_dropoff_datetime = int(row['lpep_dropoff_datetime'].timestamp() * 1_000),
            lpep_pickup_datetime = str(row['lpep_pickup_datetime']),
            lpep_dropoff_datetime = str(row['lpep_dropoff_datetime']),
            PU_locationID = int(row['PULocationID']),
            DO_locationID = int(row['DOLocationID']),
            passenger_count =
                int(passenger_count_value) if pd.notna(passenger_count_value)
                else None,
            trip_distance = float(row['trip_distance']),
            tip_amount = float(row['tip_amount']),
            total_amount = float(row['total_amount']),
        )

    @classmethod
    def deserializer(cls, data: bytes) -> Self:
        json_str = data.decode('utf-8')
        ride_dict = json.loads(json_str)
        return cls(**ride_dict)

    def serializer(self) -> bytes:
        ride_dict = dataclasses.asdict(self)
        json_str = json.dumps(ride_dict)
        return json_str.encode('utf-8')

# Producer

In [ ]:
# import time

# from kafka import KafkaProducer
# import pandas as pd

In [ ]:
data_file = 'data/green_tripdata_2025-10.parquet'
columns = [
    'lpep_pickup_datetime',
    'lpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'passenger_count',
    'trip_distance',
    'tip_amount',
    'total_amount',
]

df = (
    pd
    .read_parquet('data/green_tripdata_2025-10.parquet', columns=columns)
    .astype({'passenger_count': 'Int64'})
)

In [ ]:
topic_name = "green-trips"
producer = KafkaProducer(
    bootstrap_servers=["localhost:9092",],
    value_serializer=Ride.serializer,
)

In [ ]:
# create a ride
ride = Ride.from_row(df.iloc[0])
ride

In [ ]:
producer.send(topic=topic_name, value=ride)

In [ ]:
producer.flush()

# Consumer

In [10]:
from kafka import KafkaConsumer

In [26]:
topic_name = "green-trips"
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=["localhost:9092",],
    auto_offset_reset="earliest",
    group_id="rides-console",
    value_deserializer=Ride.deserializer,
    consumer_timeout_ms=2000,
)

In [11]:
m = next(consumer)

In [7]:
m.value.trip_distance

0.7

In [25]:
n_receipt = 0
for message in consumer:
    n_receipt += 1
    ride = message.value
    print(f"{n_receipt:03d} → dist = {ride.trip_distance:.1f}")
consumer.close()

001 → dist = 0.7
002 → dist = 1.6
003 → dist = 0.0
004 → dist = 10.4
005 → dist = 4.1
006 → dist = 7.1
007 → dist = 1.1
008 → dist = 6.0
009 → dist = 6.5
010 → dist = 0.7
011 → dist = 1.6
012 → dist = 0.0
013 → dist = 10.4
014 → dist = 4.1
015 → dist = 7.1
016 → dist = 1.1
017 → dist = 6.0
018 → dist = 6.5
019 → dist = 0.7
020 → dist = 1.6
021 → dist = 0.0
022 → dist = 10.4
023 → dist = 4.1
024 → dist = 7.1
025 → dist = 1.1
026 → dist = 6.0
027 → dist = 6.5
028 → dist = 0.7
029 → dist = 1.6
030 → dist = 0.0
031 → dist = 10.4
032 → dist = 4.1
033 → dist = 7.1
034 → dist = 1.1
035 → dist = 6.0
036 → dist = 6.5


In [27]:
count = sum(message.value.trip_distance > 5.0 for message in consumer)

print(f"{count} trips have `trip_distance` > 5")

4 trips have `trip_distance` > 5


In [28]:
consumer.close()

# Explore dataset

In [ ]:
import pandas as pd

In [ ]:
columns = [
    'lpep_pickup_datetime',
    'lpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'passenger_count',
    'trip_distance',
    'tip_amount',
    'total_amount',
]

In [ ]:
df = (
    pd
    .read_parquet('data/green_tripdata_2025-10.parquet',
                  columns=columns,
                 )
    .astype({'passenger_count': 'Int64'})
)

In [ ]:
df.head(5)

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
type(df['passenger_count'].iloc[-1])

In [ ]:
df['passenger_count'].unique()

# Brouillon

In [ ]:

@dataclass
class Val:
    x: int
    y: int | None

    @classmethod
    def from_tuple(cls, coord: tuple[int, int | None]):
        return cls(
            x = int(coord[0]),
            y = int(coord[1]) if coord[1] is not None else None
        )

In [ ]:
p1 = Val.from_tuple((2, 4))
print(p1)

In [ ]:
p2 = Val.from_tuple((1, None))
print(p2)